# Part A: held-out validation of the emotion vectors (Qwen3-8B)

DSC 180A · Member 2 (Validation and generalization) · reproduces Sofroniew et al. (2026) §1.2 on the team's Qwen3-8B vectors.

**Question.** The emotion directions from extraction run `ab1333f49024705c` were built from training stories (topics 0–13).
Do they identify the intended emotion of stories about **topics never used to build them**?

- **Inputs:** Member 1's precomputed pooled activations and vectors (notebook 03). No model, CPU only.
- **Score** for story *i*, block *L*, emotion *e*: `(pooled[i, L] - center[L]) · unit_vectors[e, L]`.
- **Validation split** (topics 14–16, 168 stories): every block, plus a label-permutation null.
- **Test split** (topics 17–19, 168 stories): evaluated **once**, at block 23 and at the best validation block.

The notebook runs in this order: data checks → unit checks → training sanity → validation → null → plots → test.
The test section reads nothing that could feed back into an earlier choice.

## 1. Configuration
`RUN_DIR` is the only path to change if the Drive archive lives somewhere else.

In [ ]:
import csv, hashlib, json, time
from pathlib import Path

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Rectangle
import sklearn
from sklearn.metrics import confusion_matrix, roc_auc_score

# Folder holding story_activations.npz and emotion_vectors.npz (unzipped Drive archive), relative to this notebook.
RUN_DIR = Path("../emotion_vector_pipeline/output_drive")
OUT_DIR = Path("outputs/A")

MAIN_LAYER = 23        # notebook 03's reference block (0-based)
N_PERM = 500           # label permutations for the null
SEED = 180             # fixed before any result was seen (the team seed from notebook 03)
FOCUS = ["desperate", "calm"]
EXPECTED_SPLIT_SIZES = {"train": 784, "validation": 168, "test": 168}

OUT_DIR.mkdir(parents=True, exist_ok=True)
print("RUN_DIR:", RUN_DIR.resolve())
print("OUT_DIR:", OUT_DIR.resolve())
print("numpy", np.__version__, "| sklearn", sklearn.__version__, "| matplotlib", matplotlib.__version__)

In [ ]:
# Plot styling: colorblind-checked categorical order, blue/gray/red diverging map, quiet axes.
INK, INK2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
BLUE, ORANGE, AQUA = "#2a78d6", "#eb6834", "#1baf7a"
DIVERGING = LinearSegmentedColormap.from_list("blue_gray_red", ["#184f95", "#6da7ec", "#f0efec", "#ec8a89", "#b3302f"])
SEQUENTIAL = LinearSegmentedColormap.from_list("blue_seq", ["#f6f9fd", "#9ec5f4", "#2a78d6", "#0d366b"])
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "savefig.facecolor": SURFACE,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2,
    "text.color": INK, "axes.titlecolor": INK, "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "lines.linewidth": 2,
    "font.size": 10, "axes.titlesize": 11, "legend.frameon": False, "figure.dpi": 110,
})

def save_fig(fig, name):
    path = OUT_DIR / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    print("saved", path)

## 2. Load and verify the archive
Shapes, keys, label order and layer order are all read from the files and checked, never assumed.

In [ ]:
ACT_PATH = RUN_DIR / "story_activations.npz"
VEC_PATH = RUN_DIR / "emotion_vectors.npz"
for p in (ACT_PATH, VEC_PATH):
    if not p.is_file():
        raise FileNotFoundError(f"Missing {p.resolve()}. Download the Drive archive "
                                "(emotion_vector_pipeline/README.md) and point RUN_DIR at it.")

with np.load(ACT_PATH, allow_pickle=False) as z:
    print("story_activations.npz keys:", z.files)
    pooled = z["pooled"]
    story_ids = z["story_ids"].astype(str)
    story_emotions = z["emotions"].astype(str)
    splits = z["splits"].astype(str)
    act_layers = z["layer_indices"].tolist()
with np.load(VEC_PATH, allow_pickle=False) as z:
    print("emotion_vectors.npz keys:  ", z.files)
    emotion_names = z["emotion_names"].astype(str).tolist()
    vec_layers = z["layer_indices"].tolist()
    unit_vectors = z["unit_vectors"]
    raw_vectors = z["raw_vectors"]
    center = z["center"]
    emotion_means = z["emotion_means"]
    saved_main = int(z["main_layer_index"]) if "main_layer_index" in z.files else None

N, L, D = pooled.shape
E = len(emotion_names)
print("pooled", pooled.shape, pooled.dtype, "| unit_vectors", unit_vectors.shape, "| center", center.shape)
print("emotion order (from file):", emotion_names)
print("layer order (from file):", act_layers)

In [ ]:
assert pooled.shape == (1120, 36, 4096) and pooled.dtype == np.float32
assert unit_vectors.shape == raw_vectors.shape == emotion_means.shape == (E, L, D) == (14, 36, 4096)
assert center.shape == (L, D)
assert act_layers == vec_layers, "Layer order differs between the two files."
assert len(set(emotion_names)) == E
assert set(story_emotions) == set(emotion_names), "Story labels do not match emotion_names."
assert len(set(story_ids)) == N, "Duplicate story IDs."
assert np.isfinite(pooled).all() and np.isfinite(unit_vectors).all()
np.testing.assert_allclose(np.linalg.norm(unit_vectors, axis=-1), 1.0, atol=1e-5)
if saved_main is not None:
    assert saved_main == MAIN_LAYER, saved_main

LAYER_POS = {layer: i for i, layer in enumerate(act_layers)}   # block index -> array axis position
MAIN_POS = LAYER_POS[MAIN_LAYER]
FOCUS_IDX = [emotion_names.index(e) for e in FOCUS]
y = np.array([emotion_names.index(e) for e in story_emotions])  # integer labels in emotion_names order

split_idx = {s: np.flatnonzero(splits == s) for s in EXPECTED_SPLIT_SIZES}
for s, n in EXPECTED_SPLIT_SIZES.items():
    assert len(split_idx[s]) == n, (s, len(split_idx[s]))
    assert (np.bincount(y[split_idx[s]], minlength=E) == n // E).all(), s
print("Split sizes:", {s: len(i) for s, i in split_idx.items()},
      "| stories per emotion:", {s: n // E for s, n in EXPECTED_SPLIT_SIZES.items()})

stories_path = RUN_DIR / "stories.jsonl"
if stories_path.is_file():
    rows = [json.loads(line) for line in stories_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    assert [r["id"] for r in rows] == story_ids.tolist(), "stories.jsonl order differs from the activations"
    assert [r["emotion"] for r in rows] == story_emotions.tolist()
    assert [r["split"] for r in rows] == splits.tolist()
    topics = {s: sorted({r["topic_id"] for r in rows if r["split"] == s}) for s in EXPECTED_SPLIT_SIZES}
    assert topics == {"train": list(range(14)), "validation": [14, 15, 16], "test": [17, 18, 19]}, topics
    print("Topic split verified against stories.jsonl:", topics)
else:
    print("stories.jsonl not found; topic cross-check skipped.")

In [ ]:
def sha256_of(path, chunk=1 << 24):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

manifest_path = RUN_DIR / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.is_file() else {}
provenance = {
    "run_id": manifest.get("run_id"),
    "model_revision": manifest.get("config", {}).get("model_revision"),
    "story_activations_sha256": sha256_of(ACT_PATH),
    "emotion_vectors_sha256": sha256_of(VEC_PATH),
    "emotion_names": emotion_names,
    "main_layer": MAIN_LAYER, "seed": SEED, "n_perm": N_PERM,
    "numpy": np.__version__, "sklearn": sklearn.__version__,
}
print(json.dumps(provenance, indent=1))

## 3. Unit checks (before trusting any number)

1. **Rebuild the vectors** from the training rows with notebook 03's recipe (per-emotion means → subtract the equal-weight mean of means → L2-normalize) and compare to the saved arrays.
2. **Score orientation:** a synthetic point at `center + 10·unit_e` must score highest on emotion *e* at every block, with score 10.
3. **Score shape and spot check** against an explicit dot product.

In [ ]:
def build_vectors(acts, labels, n_emotions):
    # Notebook 03, "Emotion directions" cell: means -> subtract mean of means -> normalise.
    means = np.stack([acts[labels == e].mean(axis=0) for e in range(n_emotions)]).astype(np.float32)
    center_ = means.mean(axis=0)
    raw = means - center_[None]
    unit = raw / np.linalg.norm(raw, axis=-1, keepdims=True)
    return means, center_, raw, unit

train_idx = split_idx["train"]
X_train, y_train = pooled[train_idx], y[train_idx]
re_means, re_center, re_raw, re_unit = build_vectors(X_train, y_train, E)

rebuild_check = {}
for name, mine, saved in [("emotion_means", re_means, emotion_means), ("center", re_center, center),
                          ("raw_vectors", re_raw, raw_vectors), ("unit_vectors", re_unit, unit_vectors)]:
    np.testing.assert_allclose(mine, saved, rtol=1e-5, atol=1e-5, err_msg=name)
    rebuild_check[name] = float(np.abs(mine - saved).max())
    print(f"{name:14s} allclose OK   max |rebuilt - saved| = {rebuild_check[name]:.3g}")

In [ ]:
def score(acts_L, center_L, unit_L):
    # acts_L (n, D), center_L (D,), unit_L (E, D) -> (n, E) projection scores
    return (acts_L - center_L) @ unit_L.T

def score_all_layers(acts, center_, unit):
    # acts (n, L, D) -> (n, L, E)
    return np.stack([score(acts[:, l], center_[l], unit[:, l]) for l in range(acts.shape[1])], axis=1)

# Orientation check on synthetic points.
syn = score_all_layers(center[None] + 10.0 * unit_vectors, center, unit_vectors)   # (E, L, E)
assert syn.shape == (E, L, E)
assert (syn.argmax(-1) == np.arange(E)[:, None]).all(), "argmax does not return the planted emotion"
np.testing.assert_allclose(syn[np.arange(E), :, np.arange(E)], 10.0, rtol=1e-4)

scores = score_all_layers(pooled, center, unit_vectors)   # (N, L, E)
assert scores.shape == (N, L, E) and np.isfinite(scores).all()
i, e = 500, emotion_names.index("calm")
explicit = float(np.dot(pooled[i, MAIN_POS].astype(np.float64) - center[MAIN_POS], unit_vectors[e, MAIN_POS]))
assert np.isclose(scores[i, MAIN_POS, e], explicit, rtol=1e-4, atol=1e-3), (scores[i, MAIN_POS, e], explicit)
print("Synthetic orientation check OK | scores shape", scores.shape, "| spot check OK")

## 4. Metrics

- **Top-1 accuracy:** the argmax over the 14 scores equals the intended label (chance = 1/14).
- **One-vs-rest AUROC** for emotion *e*: that emotion's score separates its stories from the other 13 emotions' stories (chance = 0.5).

The null loop uses a vectorized rank (Mann–Whitney) AUROC, where tied scores share their average rank. It is checked against `sklearn.metrics.roc_auc_score` on a synthetic case with ties (here) and on the real validation scores (§6).

In [ ]:
CHANCE = 1 / E

def top1_accuracy(s, labels):          # s: (n, E)
    return float((s.argmax(-1) == labels).mean())

def ovr_auroc(s, labels):              # sklearn reference, s: (n, E) -> (E,)
    return np.array([roc_auc_score(labels == e, s[:, e]) for e in range(s.shape[-1])])

def average_ranks(s):
    # 1-based ranks along axis 0; tied values share their average rank (as in Mann-Whitney / sklearn).
    n = s.shape[0]
    order = s.argsort(axis=0, kind="stable")
    srt = np.take_along_axis(s, order, axis=0)
    pos = np.arange(n).reshape((n,) + (1,) * (s.ndim - 1)) + np.zeros_like(srt, dtype=np.int64)
    starts = np.ones(srt.shape, bool); starts[1:] = srt[1:] != srt[:-1]
    ends = np.ones(srt.shape, bool); ends[:-1] = srt[1:] != srt[:-1]
    first = np.maximum.accumulate(np.where(starts, pos, 0), axis=0)
    last = np.flip(np.minimum.accumulate(np.flip(np.where(ends, pos, n - 1), 0), axis=0), 0)
    ranks = np.empty(srt.shape, np.float64)
    np.put_along_axis(ranks, order, (first + last) / 2 + 1, axis=0)
    return ranks

def ovr_auroc_fast(s, labels):
    # Mann-Whitney AUROC along axis 0 for every (layer, emotion) column; s: (n, L, E) -> (L, E).
    ranks = average_ranks(s)
    onehot = np.eye(s.shape[-1])[labels]                       # (n, E)
    n_pos = onehot.sum(0)                                      # (E,)
    rank_sum = np.einsum("nle,ne->le", ranks, onehot)
    return (rank_sum - n_pos * (n_pos + 1) / 2) / (n_pos * (len(labels) - n_pos))

# Unit test with many forced ties (scores rounded to 1 decimal): must equal sklearn exactly.
_rng = np.random.default_rng(0)
_lab = np.repeat(np.arange(E), 12)
_s = np.round(_rng.normal(size=(len(_lab), 3, E)), 1)
assert (np.diff(np.sort(_s, axis=0), axis=0) == 0).any(), "test should contain ties"
np.testing.assert_allclose(ovr_auroc_fast(_s, _lab), np.stack([ovr_auroc(_s[:, l], _lab) for l in range(3)]), atol=1e-12)
print("tie-aware fast AUROC matches sklearn on a synthetic case with ties")

## 5. Training-split sanity check (in-sample; these stories built the vectors)
Well above chance is expected here. If it isn't, something upstream is wrong. This is **not** a generalization result.

In [ ]:
train_scores = scores[train_idx]
train_acc = np.array([top1_accuracy(train_scores[:, l], y_train) for l in range(L)])
print(f"chance = {CHANCE:.3f}")
print(f"train top-1 at block {MAIN_LAYER}: {train_acc[MAIN_POS]:.3f} | min over blocks {train_acc.min():.3f} | max {train_acc.max():.3f}")
assert train_acc[MAIN_POS] > 5 * CHANCE, "Training accuracy is not well above chance; stop and investigate."

## 6. Validation split: all 36 blocks

**Best validation block** (chosen here, before the test split is touched): the highest validation top-1 accuracy, with ties broken by higher macro-AUROC and then by the lower block index.

In [ ]:
val_idx = split_idx["validation"]
val_scores, y_val = scores[val_idx], y[val_idx]
val_acc = np.array([top1_accuracy(val_scores[:, l], y_val) for l in range(L)])
val_auroc = np.stack([ovr_auroc(val_scores[:, l], y_val) for l in range(L)])     # (L, E), sklearn
val_macro = val_auroc.mean(axis=1)

# Check the fast AUROC used in the null loop against sklearn on the real data.
np.testing.assert_allclose(ovr_auroc_fast(val_scores, y_val), val_auroc, rtol=0, atol=1e-12)
print("fast AUROC == sklearn AUROC on all", L * E, "validation (block, emotion) columns")

BEST_POS = sorted(range(L), key=lambda l: (-val_acc[l], -val_macro[l], act_layers[l]))[0]
BEST_LAYER = act_layers[BEST_POS]
print(f"block {MAIN_LAYER}: val top-1 {val_acc[MAIN_POS]:.3f}, macro AUROC {val_macro[MAIN_POS]:.3f}")
print(f"best validation block: {BEST_LAYER} (top-1 {val_acc[BEST_POS]:.3f}, macro AUROC {val_macro[BEST_POS]:.3f})")
for name, k in zip(FOCUS, FOCUS_IDX):
    print(f"  {name:10s} AUROC  block {MAIN_LAYER}: {val_auroc[MAIN_POS, k]:.3f} | block {BEST_LAYER}: {val_auroc[BEST_POS, k]:.3f}")

## 7. Null control: label permutation

Shuffle the emotion labels among the **training** stories (keeping 56 per emotion), rebuild the vectors with the same `build_vectors` checked in §3, and rescore the validation split.
There are 500 permutations with fixed seed 180. A real metric counts as beyond the null when it exceeds the null's 95th percentile. The p-value is `(1 + #{null ≥ real}) / (1 + 500)`.

As a code-path check, running the null function on the unshuffled labels must reproduce the real validation metrics.

In [ ]:
X_val = pooled[val_idx]

def null_metrics(train_labels):
    _, c, _, u = build_vectors(X_train, train_labels, E)
    s = score_all_layers(X_val, c, u)                                         # (168, L, E)
    acc = (s.argmax(-1) == y_val[:, None]).mean(0)                            # (L,)
    return acc, ovr_auroc_fast(s, y_val)                                      # (L,), (L, E)

acc_id, auc_id = null_metrics(y_train)
np.testing.assert_allclose(acc_id, val_acc, atol=1e-12)
np.testing.assert_allclose(auc_id, val_auroc, atol=1e-6)
print("null code path with unshuffled labels reproduces the real validation metrics")

rng = np.random.default_rng(SEED)
null_acc = np.empty((N_PERM, L))
null_auroc = np.empty((N_PERM, L, E))
t0 = time.time()
for p in range(N_PERM):
    null_acc[p], null_auroc[p] = null_metrics(rng.permutation(y_train))
    if (p + 1) % 100 == 0:
        print(f"  {p + 1}/{N_PERM} permutations, {time.time() - t0:.0f}s")

null_macro = null_auroc.mean(-1)                                  # (P, L)
acc_p95, macro_p95 = np.percentile(null_acc, 95, axis=0), np.percentile(null_macro, 95, axis=0)
auroc_p95 = np.percentile(null_auroc, 95, axis=0)                 # (L, E)
p_value = lambda null, real: (1 + (null >= real).sum(0)) / (1 + N_PERM)
acc_p, macro_p, auroc_p = p_value(null_acc, val_acc), p_value(null_macro, val_macro), p_value(null_auroc, val_auroc)

for pos in sorted({MAIN_POS, BEST_POS}):
    print(f"block {act_layers[pos]}: val top-1 {val_acc[pos]:.3f} vs null p95 {acc_p95[pos]:.3f} (null mean {null_acc[:, pos].mean():.3f}), p = {acc_p[pos]:.4f}")
    print(f"          macro AUROC {val_macro[pos]:.3f} vs null p95 {macro_p95[pos]:.3f}, p = {macro_p[pos]:.4f}")
print("blocks where val top-1 > null p95:", int((val_acc > acc_p95).sum()), "of", L)

## 8. Validation tables and figures

In [ ]:
def write_csv(path, header, rows_):
    with open(path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(header); w.writerows(rows_)
    print("saved", path)

header = (["block", "train_top1", "val_top1", "null_top1_mean", "null_top1_p95", "top1_p",
           "val_macro_auroc", "null_macro_p95", "macro_p"]
          + [f"auroc_{e}" for e in emotion_names] + [f"null_p95_auroc_{e}" for e in emotion_names])
write_csv(OUT_DIR / "validation_by_block.csv", header, [
    [act_layers[l], round(train_acc[l], 4), round(val_acc[l], 4), round(null_acc[:, l].mean(), 4),
     round(acc_p95[l], 4), round(acc_p[l], 4), round(val_macro[l], 4), round(macro_p95[l], 4), round(macro_p[l], 4)]
    + [round(v, 4) for v in val_auroc[l]] + [round(v, 4) for v in auroc_p95[l]] for l in range(L)])

for pos in sorted({MAIN_POS, BEST_POS}):
    write_csv(OUT_DIR / f"validation_auroc_vs_null_block{act_layers[pos]}.csv",
              ["emotion", "val_auroc", "null_p95", "p", "above_null_p95"],
              [[e, round(val_auroc[pos, k], 4), round(auroc_p95[pos, k], 4), round(auroc_p[pos, k], 4),
                bool(val_auroc[pos, k] > auroc_p95[pos, k])] for k, e in enumerate(emotion_names)])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
ax.fill_between(act_layers, np.percentile(null_acc, 5, axis=0), acc_p95, color=GRID, alpha=0.9, lw=0,
                label="label-shuffle null (5th–95th pct)")
ax.plot(act_layers, train_acc, color=MUTED, lw=1.5, ls="--", label="train (in-sample)")
ax.plot(act_layers, val_acc, color=BLUE, label="validation (held-out topics)")
ax.axhline(CHANCE, color=MUTED, lw=1, ls=":")
ax.text(act_layers[-1], CHANCE + 0.01, "chance 1/14", color=INK2, ha="right", va="bottom", fontsize=9)
ax.axvline(MAIN_LAYER, color=INK2, lw=1)
ax.text(MAIN_LAYER + 0.3, 0.98, f"block {MAIN_LAYER}", color=INK2, va="top", fontsize=9)
ax.plot([BEST_LAYER], [val_acc[BEST_POS]], "o", ms=8, color=BLUE, mec=SURFACE, mew=2)
ax.annotate(f"best val: block {BEST_LAYER}\n{val_acc[BEST_POS]:.2f}", (BEST_LAYER, val_acc[BEST_POS]),
            xytext=(0, -34), textcoords="offset points", ha="center", fontsize=9, color=INK2)
ax.set(xlabel="decoder block (0-based)", ylabel="14-way top-1 accuracy", ylim=(0, 1.02), xlim=(-0.5, L - 0.5),
       title="Emotion-vector classification of stories by block (Qwen3-8B, validation split)")
ax.legend(loc="upper left", fontsize=9)
save_fig(fig, "val_accuracy_by_block.png"); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.2))
for name, k, col in [(FOCUS[0], FOCUS_IDX[0], BLUE), (FOCUS[1], FOCUS_IDX[1], ORANGE)]:
    ax.plot(act_layers, val_auroc[:, k], color=col, label=f"{name}")
    ax.plot(act_layers, auroc_p95[:, k], color=col, lw=1, ls="--", alpha=0.8, label=f"{name}: null p95")
ax.plot(act_layers, val_macro, color=MUTED, lw=1.5, label="mean of 14 emotions")
ax.axhline(0.5, color=MUTED, lw=1, ls=":")
ax.axvline(MAIN_LAYER, color=INK2, lw=1)
ax.text(MAIN_LAYER + 0.3, 0.42, f"block {MAIN_LAYER}", color=INK2, fontsize=9)
ax.set(xlabel="decoder block (0-based)", ylabel="one-vs-rest AUROC", ylim=(0.4, 1.01), xlim=(-0.5, L - 0.5),
       title="Validation AUROC by block: desperate and calm")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.16), fontsize=9, ncol=3)
save_fig(fig, "val_auroc_by_block_desperate_calm.png"); plt.show()

In [ ]:
for pos in sorted({MAIN_POS, BEST_POS}):
    order = np.argsort(val_auroc[pos])
    fig, ax = plt.subplots(figsize=(6.5, 4.8))
    colors = [BLUE if emotion_names[k] in FOCUS else "#b7b6ae" for k in order]
    ax.barh(range(E), val_auroc[pos, order] - 0.5, left=0.5, color=colors, height=0.7)
    ax.scatter(auroc_p95[pos, order], range(E), marker="|", s=160, color=INK, zorder=3, label="null p95")
    for r, k in enumerate(order):
        ax.text(val_auroc[pos, k] + 0.005, r, f"{val_auroc[pos, k]:.2f}", va="center", fontsize=8, color=INK2)
    ax.set_yticks(range(E), [emotion_names[k] for k in order])
    ax.axvline(0.5, color=MUTED, lw=1, ls=":")
    ax.set(xlim=(0.45, 1.04), xlabel="one-vs-rest AUROC (validation)",
           title=f"Per-emotion validation AUROC, block {act_layers[pos]}")
    ax.grid(axis="y", visible=False); ax.legend(loc="lower right", fontsize=9)
    save_fig(fig, f"val_auroc_per_emotion_block{act_layers[pos]}.png"); plt.show()

fig, ax = plt.subplots(figsize=(6.5, 3.2))
n_val = len(val_idx)   # accuracy moves in steps of 1/168, so bins sit on those steps
ax.hist(null_acc[:, MAIN_POS], bins=(np.arange(0, round(null_acc[:, MAIN_POS].max() * n_val) + 2) - 0.5) / n_val,
        color="#b7b6ae", ec=SURFACE, lw=0.5)
ax.axvline(acc_p95[MAIN_POS], color=INK2, ls="--", lw=1)
ax.text(acc_p95[MAIN_POS], ax.get_ylim()[1] * 0.95, " null p95", color=INK2, fontsize=9, va="top")
ax.axvline(val_acc[MAIN_POS], color=BLUE)
ax.text(val_acc[MAIN_POS], ax.get_ylim()[1] * 0.95, f"real {val_acc[MAIN_POS]:.2f} ", color=BLUE, ha="right", fontsize=9, va="top")
ax.set(xlabel="validation top-1 accuracy", ylabel="permutations",
       title=f"Label-shuffle null, block {MAIN_LAYER} ({N_PERM} permutations)")
ax.grid(axis="x", visible=False)
save_fig(fig, f"null_accuracy_hist_block{MAIN_LAYER}.png"); plt.show()

## 9. Test split: evaluated once

Evaluated at block 23 and at the best validation block. Both were fixed in §6, before this section.
`test_lock.json` records those blocks the first time the test split is scored. A rerun that would score different blocks stops with an error.

**Heatmap:** rows = true emotion, columns = emotion vector. Each cell is the mean test score, z-scored per vector column with that column's validation-split mean and standard deviation (population SD over all 168 validation stories).

In [ ]:
TEST_LAYERS = [MAIN_LAYER] + ([BEST_LAYER] if BEST_LAYER != MAIN_LAYER else [])
lock_path = OUT_DIR / "test_lock.json"
if lock_path.is_file():
    locked = json.loads(lock_path.read_text())
    assert locked["test_layers"] == TEST_LAYERS, f"Test blocks changed after the test split was first scored: {locked}"
    print("test_lock.json found; same blocks as the first evaluation:", locked)
else:
    lock_path.write_text(json.dumps({"test_layers": TEST_LAYERS,
                                     "first_evaluated_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}, indent=2))
    print("wrote", lock_path)

test_idx = split_idx["test"]
test_scores, y_test = scores[test_idx], y[test_idx]
test_results = {}
for layer in TEST_LAYERS:
    pos = LAYER_POS[layer]
    s = test_scores[:, pos]
    pred = s.argmax(-1)
    auroc = ovr_auroc(s, y_test)
    cm = confusion_matrix(y_test, pred, labels=range(E))
    mu, sd = val_scores[:, pos].mean(0), val_scores[:, pos].std(0)
    z = (s - mu) / sd
    heat = np.stack([z[y_test == t].mean(0) for t in range(E)])           # (true, vector)
    test_results[layer] = dict(top1=top1_accuracy(s, y_test), auroc=auroc, cm=cm, heat=heat)
    print(f"TEST block {layer}: top-1 {test_results[layer]['top1']:.3f} (chance {CHANCE:.3f}), "
          f"macro AUROC {auroc.mean():.3f}, " + ", ".join(f"{e} {auroc[k]:.3f}" for e, k in zip(FOCUS, FOCUS_IDX)))
    write_csv(OUT_DIR / f"test_confusion_block{layer}.csv", ["true \\ predicted"] + emotion_names,
              [[emotion_names[t]] + cm[t].tolist() for t in range(E)])
    write_csv(OUT_DIR / f"test_heatmap_zscore_block{layer}.csv", ["true \\ vector"] + emotion_names,
              [[emotion_names[t]] + [round(v, 4) for v in heat[t]] for t in range(E)])

In [ ]:
def outline_diagonal(ax, n):
    for k in range(n):
        ax.add_patch(Rectangle((k - 0.5, k - 0.5), 1, 1, fill=False, ec=INK, lw=1.2))

for layer, res in test_results.items():
    fig, axes = plt.subplots(1, 2, figsize=(14, 6.2))
    ax = axes[0]
    im = ax.imshow(res["cm"], cmap=SEQUENTIAL, vmin=0, vmax=12)
    for t in range(E):
        for p_ in range(E):
            if res["cm"][t, p_]:
                ax.text(p_, t, res["cm"][t, p_], ha="center", va="center", fontsize=8,
                        color="#ffffff" if res["cm"][t, p_] >= 7 else INK)
    ax.set_title(f"Confusion (counts of 12), top-1 {res['top1']:.2f}")
    ax.set_xlabel("predicted (argmax score)"); ax.set_ylabel("true emotion")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    ax = axes[1]
    vmax = float(np.abs(res["heat"]).max())
    im = ax.imshow(res["heat"], cmap=DIVERGING, vmin=-vmax, vmax=vmax)
    outline_diagonal(ax, E)
    ax.set_title("Mean test score, z-scored per vector (validation mean/SD)")
    ax.set_xlabel("emotion vector"); ax.set_ylabel("true emotion")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="z")
    for ax in axes:
        ax.set_xticks(range(E), emotion_names, rotation=60, ha="right")
        ax.set_yticks(range(E), emotion_names)
        ax.grid(False)
    fig.suptitle(f"Test split (topics 17–19, n=168), block {layer}", fontsize=12)
    fig.tight_layout()
    save_fig(fig, f"test_confusion_heatmap_block{layer}.png"); plt.show()

## 10. Summary file

In [ ]:
summary = {
    "provenance": provenance,
    "inputs": {"run_dir": str(RUN_DIR), "story_activations": str(ACT_PATH), "emotion_vectors": str(VEC_PATH)},
    "unit_checks": {"vector_rebuild_max_abs_diff": rebuild_check, "synthetic_orientation": "pass",
                    "fast_auroc_matches_sklearn": "pass", "null_identity_reproduces_real": "pass"},
    "chance_top1": CHANCE,
    "train_in_sample_top1": {"block_%d" % MAIN_LAYER: train_acc[MAIN_POS], "min": train_acc.min(), "max": train_acc.max()},
    "best_validation_block": BEST_LAYER,
    "validation": {f"block_{act_layers[pos]}": {
        "top1": val_acc[pos], "null_top1_mean": null_acc[:, pos].mean(), "null_top1_p95": acc_p95[pos], "top1_p": acc_p[pos],
        "macro_auroc": val_macro[pos], "null_macro_p95": macro_p95[pos], "macro_p": macro_p[pos],
        "auroc": dict(zip(emotion_names, val_auroc[pos])), "null_auroc_p95": dict(zip(emotion_names, auroc_p95[pos])),
        "auroc_p": dict(zip(emotion_names, auroc_p[pos]))} for pos in sorted({MAIN_POS, BEST_POS})},
    "validation_blocks_top1_above_null_p95": int((val_acc > acc_p95).sum()),
    "test": {f"block_{layer}": {"top1": r["top1"], "macro_auroc": r["auroc"].mean(),
                               "auroc": dict(zip(emotion_names, r["auroc"])),
                               "diagonal_is_row_max_in_heatmap": int((r["heat"].argmax(1) == np.arange(E)).sum())}
             for layer, r in test_results.items()},
}
def _py(o):
    if isinstance(o, dict): return {k: _py(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_py(v) for v in o]
    if isinstance(o, (np.floating, float)): return round(float(o), 5)
    if isinstance(o, np.integer): return int(o)
    return o
(OUT_DIR / "summary.json").write_text(json.dumps(_py(summary), indent=2))
print("saved", OUT_DIR / "summary.json")
print(json.dumps(_py({k: summary[k] for k in ["chance_top1", "best_validation_block", "validation_blocks_top1_above_null_p95", "test"]}), indent=1)[:3000])